In [1]:
import json
with open("documents.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

In [2]:
len(documents)

1368

In [3]:
import numpy as np
X = np.load("embeddings.npy")

In [4]:
X.shape

(1368, 384)

In [5]:
from sentence_transformers import SentenceTransformer
MODEL_PATH = "./models/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_PATH)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
model

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

---

In [7]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [13]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [14]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'doc_id': '41aabbd7c5'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Cour

In [15]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [16]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

---

In [10]:
!wget -q https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
!wget -q https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [11]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [12]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, you can still join. If you want a certificate, you need to submit your project while submissions are still being accepted.'

In [9]:
from rag_helper import RAGVector
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [10]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still sign up and start learning. If you want a certificate, make sure to submit your project while submissions are still being accepted.'